# Stage 2 -- Aggregation: Daily Full Moments

## Input
- `Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/panel_stock_daily_engineered.parquet` -- Panel A, ~100 stocks × ~4,656 dates × 192 factors, keyed on `(permno, date)`
- `Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/panel_macro_daily_engineered.parquet` -- Panel C, market-level macro daily factors, keyed on `date`

## Purpose
Extends the daily aggregation from Notebook 01 by computing **five** cap-weighted cross-sectional statistics per stock factor instead of just the mean. The five moments capture how the distribution of each factor across ~100 stocks changes over time, not just the average level. For example: `cwstd(turnover)` measures how dispersed trading activity is across stocks today; `cwskew(dlyretx)` captures whether the cross-section of returns is positively or negatively skewed; `spread(bid_ask_spread)` shows how different the most vs least liquid stocks are.

Steps 1--4 are identical to Notebook 01 (load & trim, handle warmup NaN, winsorise, compute target). Step 5 is the key difference: full moments aggregation.

---

## Pipeline

### Steps 1--4: Load, Trim, NaN, Winsorise, Target
Identical to Notebook 01. Panel A trimmed to 2006-07-03+, NaN left in place for per-factor per-date exclusion, all stock factors cast to `float64` and winsorised at 1st/99th percentile cross-sectionally per date using vectorised `groupby.transform`. Three ISO columns dropped post-winsorisation. Target computed as next-day cap-weighted market return with date-gap guard.

### Step 5: Cap-Weighted Full Moments Aggregation
For each of the ~189 surviving stock factor columns, five statistics are computed per date. All computation uses numpy arrays and pandas `groupby('date').sum()` (C-engine) to avoid Python loops over dates. Progress is printed every 25 factors with elapsed time and remaining estimate.

For each factor the following are computed in sequence:

**cwmean** (cap-weighted mean):
- Valid mask: non-NaN in both cap and factor value
- Weighted sum / cap sum per date

**cwstd** (cap-weighted standard deviation):
- Deviations from cwmean are computed per stock
- `sqrt(Σ(w × dev²) / Σ(w))` per date
- Tiny negative values from floating-point noise clamped to zero before sqrt

**cwskew** (cap-weighted skewness):
- Standardised deviations (z-scores) computed using cwstd; stocks where cwstd < 1e-10 are excluded
- `Σ(w × z³) / Σ(w)` per date

**cwkurt** (cap-weighted kurtosis):
- `Σ(w × z⁴) / Σ(w)` per date (raw kurtosis, not excess; normal distribution = 3.0)

**spread** (p90 - p10, unweighted):
- 90th minus 10th percentile of the raw factor values across stocks per date, computed without cap weighting
- Captures the range between the most extreme stocks

Results for all five moments × all factors are assembled into a single DataFrame in one shot. NaN counts in the aggregated output are reported.

### Step 6: Merge with Macro Daily + Target
Column name conflicts between aggregated stock moments and Panel C macro factors are checked and resolved with a `stock_` prefix if needed. Three-way merge on `date` (inner join with Panel C, left join for target).


**Warmup trim:** First 50 rows dropped for Panel C rolling feature warmup. Last row dropped (no target available).

### Step 7: Validation
- No duplicate dates
- NaN count across all feature columns
- Target NaN count
- **Target integrity check:** correlation between `target_daily_return` on date t and `dlyretx_cwmean` on date t+1 (should be ~0.99+)
- **Moment sanity checks:**
  - Minimum `cwstd` across all factors and dates (should be ≥ 0)
  - Minimum and mean `cwkurt` (raw kurtosis; mean ~3 for normal-like distributions, >3 indicates fat tails)
  - Minimum `spread` (should be ≥ 0)
- Column breakdown: stock moment columns, macro factors, target, date

### Step 8: Save
Sorted by date and saved to parquet.

---

## Key Design Decisions
- **Five moments per factor** rather than just the mean, to capture distributional dynamics across the ~100-stock cross-section.
- **cwkurt is raw kurtosis** (not excess), so a normal distribution scores ~3.0. Values above 3 indicate fat tails in the cross-section.
- **spread is unweighted** (p90 - p10 of raw values), in contrast to the four cap-weighted moments. This captures extreme-stock behaviour regardless of market cap.
- **Vectorised aggregation:** for each factor, five separate `groupby().sum()` calls are made, each operating on pre-computed numpy arrays. This avoids Python loops over dates while computing all five moments correctly.
- All other design decisions (start date, winsorisation, ISO drops, 50-row warmup trim, target computation) are identical to Notebook 01.

## Output
`Data/Data_Collection/Final/Stage_2/agg_market_daily_full_moments.parquet` -- keyed on `date`, containing five cap-weighted cross-sectional moments (cwmean, cwstd, cwskew, cwkurt, spread) for each stock daily factor, plus all Panel C macro factors and `target_daily_return`

In [ ]:
# %% [markdown]
# # Stage 2 — Aggregation: Daily Full Moments
#
# Same pipeline as Notebook 01 but computes FIVE cap-weighted cross-sectional
# statistics per stock factor instead of just the mean:
#   - cwmean:  cap-weighted mean
#   - cwstd:   cap-weighted standard deviation
#   - cwskew:  cap-weighted skewness
#   - cwkurt:  cap-weighted kurtosis
#   - spread:  p90 - p10 (unweighted cross-sectional range)
#
# These capture how the DISTRIBUTION of each factor across ~100 stocks
# changes over time, not just the average. For example:
#   - cwstd(turnover) = how dispersed is trading activity across stocks today
#   - cwskew(dlyretx) = is the cross-section of returns positively or negatively skewed
#   - spread(bid_ask_spread) = how different are the most vs least liquid stocks
#
# Input:
#   Panel A: Stage_1_5/.../panel_stock_daily_engineered.parquet
#   Panel C: Stage_1_5/.../panel_macro_daily_engineered.parquet
#
# Output:
#   Stage_2/agg_market_daily_full_moments.parquet

# %%
import pandas as pd
import numpy as np
from pathlib import Path
import time

PANEL_A_PATH = Path('../../../Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/panel_stock_daily_engineered.parquet')
PANEL_C_PATH = Path('../../../Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/panel_macro_daily_engineered.parquet')
OUT_DIR = Path('../../../Data/Data_Collection/Final/Stage_2')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 1: LOAD & TRIM (identical to Notebook 01)
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STEP 1: LOAD & TRIM")
print("=" * 90)

START_DATE = '2004-01-02'
MIN_STOCKS = 50
panel_a = pd.read_parquet(PANEL_A_PATH)
panel_a['date'] = pd.to_datetime(panel_a['date'])
print(f"\n  Panel A loaded: {panel_a.shape[0]:,} rows × {panel_a.shape[1]} columns")

panel_a = panel_a[panel_a['date'] >= START_DATE].reset_index(drop=True)
print(f"  After trim:  {panel_a.shape[0]:,} rows")
print(f"  Date range: {panel_a['date'].min().date()} → {panel_a['date'].max().date()}")
print(f"  Unique dates: {panel_a['date'].nunique():,}")
print(f"  Avg stocks/date: {panel_a.groupby('date').size().mean():.1f}")

panel_c = pd.read_parquet(PANEL_C_PATH)
panel_c['date'] = pd.to_datetime(panel_c['date'])
print(f"\n  Panel C loaded: {panel_c.shape[0]:,} rows × {panel_c.shape[1]} columns")

meta_cols = ['permno', 'date', 'dlyret', 'dlycap']
factor_cols = [c for c in panel_a.columns if c not in meta_cols]
macro_factor_cols = [c for c in panel_c.columns if c != 'date']

print(f"\n  Panel A factor columns: {len(factor_cols)}")
print(f"  Panel C factor columns: {len(macro_factor_cols)}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2: HANDLE WARMUP NaN (identical to Notebook 01)
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 2: HANDLE WARMUP NaN")
print("=" * 90)

nan_before = panel_a[factor_cols].isna().sum()
nan_factors = nan_before[nan_before > 0]
print(f"\n  Factors with NaN: {len(nan_factors)} / {len(factor_cols)}")
print(f"  Total NaN cells: {nan_before.sum():,}")
print(f"  Strategy: NaN stocks excluded per-factor per-date during aggregation")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 3: WINSORISE (identical to Notebook 01)
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 3: WINSORISE STOCK FACTORS (1st/99th per date)")
print("=" * 90)

t0 = time.time()

for c in ['permno', 'date', 'dlyret', 'dlycap']:
    assert c not in factor_cols, f"FATAL: {c} is in factor_cols!"

panel_a[factor_cols] = panel_a[factor_cols].astype('float64')

for col in factor_cols:
    p01 = panel_a.groupby('date')[col].transform('quantile', 0.01)
    p99 = panel_a.groupby('date')[col].transform('quantile', 0.99)
    panel_a[col] = panel_a[col].clip(lower=p01, upper=p99)

elapsed = time.time() - t0
print(f"\n  Winsorised {len(factor_cols)} factors in {elapsed:.1f}s")
print(f"  ✓ Winsorisation complete")

# Drop ISO columns (late-starting, redundant)
iso_drop = ['iso_dollar_to_cap', 'iso_vol_to_shrout', 'n_iso_trade_pct']
iso_drop = [c for c in iso_drop if c in factor_cols]
panel_a = panel_a.drop(columns=iso_drop)
factor_cols = [c for c in factor_cols if c not in iso_drop]
print(f"  Dropped {len(iso_drop)} ISO columns")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 4: COMPUTE TARGET (identical to Notebook 01)
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 4: COMPUTE TARGET (next-day cap-weighted market return)")
print("=" * 90)

panel_a = panel_a.sort_values(['permno', 'date']).reset_index(drop=True)
panel_a['next_day_ret'] = panel_a.groupby('permno')['dlyret'].shift(-1)

date_diff = panel_a.groupby('permno')['date'].diff(-1).abs()
panel_a.loc[date_diff > pd.Timedelta(days=5), 'next_day_ret'] = np.nan

target_df = panel_a.dropna(subset=['next_day_ret', 'dlycap']).copy()
target_df['weighted_next_ret'] = target_df['dlycap'] * target_df['next_day_ret']

target_agg = target_df.groupby('date').agg(
    target_daily_return=('weighted_next_ret', 'sum'),
    total_cap=('dlycap', 'sum'),
).reset_index()
target_agg['target_daily_return'] = target_agg['target_daily_return'] / target_agg['total_cap']
target_agg = target_agg[['date', 'target_daily_return']]

print(f"\n  Target computed: {len(target_agg):,} dates")
print(f"    Mean: {target_agg['target_daily_return'].mean():.6f}")
print(f"    Std:  {target_agg['target_daily_return'].std():.6f}")

last_date = panel_a['date'].max()
last_target = target_agg[target_agg['date'] == last_date]['target_daily_return']
if len(last_target) == 0 or last_target.isna().all():
    print(f"  ✓ Last date ({last_date.date()}) correctly has no target")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 5: CAP-WEIGHTED FULL MOMENTS AGGREGATION
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 5: CAP-WEIGHTED FULL MOMENTS AGGREGATION")
print("=" * 90)

t0 = time.time()

agg_factors = factor_cols.copy()
cap_arr = panel_a['dlycap'].to_numpy(dtype='float64', na_value=np.nan)
date_arr = panel_a['date'].values
sorted_dates = np.sort(panel_a['date'].unique())
n_dates = len(sorted_dates)

# Pre-compute date group indices for fast lookup
date_to_idx = {d: i for i, d in enumerate(sorted_dates)}
row_date_idx = np.array([date_to_idx[d] for d in date_arr])

print(f"\n  Computing 5 statistics × {len(agg_factors)} factors × {n_dates:,} dates...")
print(f"  (cwmean, cwstd, cwskew, cwkurt, spread)\n")

# Storage for all results
all_results = {}

for i, col in enumerate(agg_factors):
    vals = panel_a[col].to_numpy(dtype='float64', na_value=np.nan)
    
    # Mask: valid where both cap and factor are non-NaN
    valid = ~(np.isnan(vals) | np.isnan(cap_arr))
    
    # ── cwmean ───────────────────────────────────────────────────────────
    weighted = np.where(valid, cap_arr * vals, 0.0)
    cap_valid = np.where(valid, cap_arr, 0.0)
    
    temp = pd.DataFrame({'date': date_arr, 'wv': weighted, 'wc': cap_valid,
                         'n': valid.astype('int64')})
    agg = temp.groupby('date', sort=True).sum()
    cwmean_per_date = (agg['wv'] / agg['wc'].replace(0, np.nan)).values
    # Gate all five moments on the same stock count. Applied when STORING, not
    # to the variable -- cwmean_per_date and cwstd_per_date are needed at full
    # precision below for the deviation and standardisation steps.
    enough = agg['n'].values >= MIN_STOCKS
    all_results[f'{col}_cwmean'] = np.where(enough, cwmean_per_date, np.nan)
    
    # Map cwmean back to stock level for deviation computation
    cwmean_mapped = cwmean_per_date[row_date_idx]
    
    # ── Deviations ───────────────────────────────────────────────────────
    dev = np.where(valid, vals - cwmean_mapped, 0.0)
    
    # ── cwstd ────────────────────────────────────────────────────────────
    # sqrt(Σ(w × dev²) / Σ(w))
    w_dev_sq = np.where(valid, cap_arr * dev * dev, 0.0)
    
    temp_std = pd.DataFrame({'date': date_arr, 'wds': w_dev_sq, 'wc': cap_valid})
    agg_std = temp_std.groupby('date', sort=True).sum()
    
    cwvar = (agg_std['wds'] / agg_std['wc'].replace(0, np.nan)).values
    cwstd_per_date = np.sqrt(np.maximum(cwvar, 0))  # clamp tiny negatives from float noise
    all_results[f'{col}_cwstd'] = np.where(enough, cwstd_per_date, np.nan)
    
    # Map cwstd back for standardisation
    cwstd_mapped = cwstd_per_date[row_date_idx]
    
    # Standardised deviations (z-scores within each date)
    safe_std = np.where(cwstd_mapped > 1e-10, cwstd_mapped, np.nan)
    z = np.where(valid, dev / safe_std, 0.0)
    z_valid = valid & ~np.isnan(safe_std)
    
    # ── cwskew ───────────────────────────────────────────────────────────
    # Σ(w × z³) / Σ(w)
    w_z3 = np.where(z_valid, cap_arr * z * z * z, 0.0)
    cap_z_valid = np.where(z_valid, cap_arr, 0.0)
    
    temp_skew = pd.DataFrame({'date': date_arr, 'wz3': w_z3, 'wc': cap_z_valid})
    agg_skew = temp_skew.groupby('date', sort=True).sum()
    
    cwskew_per_date = (agg_skew['wz3'] / agg_skew['wc'].replace(0, np.nan)).values
    all_results[f'{col}_cwskew'] = np.where(enough, cwskew_per_date, np.nan)
    
    # ── cwkurt ───────────────────────────────────────────────────────────
    # Σ(w × z⁴) / Σ(w)
    w_z4 = np.where(z_valid, cap_arr * z * z * z * z, 0.0)
    
    temp_kurt = pd.DataFrame({'date': date_arr, 'wz4': w_z4, 'wc': cap_z_valid})
    agg_kurt = temp_kurt.groupby('date', sort=True).sum()
    
    cwkurt_per_date = (agg_kurt['wz4'] / agg_kurt['wc'].replace(0, np.nan)).values
    all_results[f'{col}_cwkurt'] = np.where(enough, cwkurt_per_date, np.nan)
    
    # ── spread (p90 - p10, unweighted) ───────────────────────────────────
    col_series = pd.Series(vals, name='val')
    date_series = pd.Series(date_arr, name='date')
    temp_spread = pd.DataFrame({'date': date_series, 'val': col_series})
    
    p90 = temp_spread.groupby('date')['val'].quantile(0.90)
    p10 = temp_spread.groupby('date')['val'].quantile(0.10)
    spread_per_date = (p90 - p10).values
    all_results[f'{col}_spread'] = np.where(enough, spread_per_date, np.nan)
    
    if (i + 1) % 25 == 0:
        elapsed_so_far = time.time() - t0
        rate = (i + 1) / elapsed_so_far
        remaining = (len(agg_factors) - i - 1) / rate
        print(f"    {i + 1}/{len(agg_factors)} factors done... "
              f"({elapsed_so_far:.0f}s elapsed, ~{remaining:.0f}s remaining)")

# Build result DataFrame in one shot
all_results['date'] = sorted_dates
agg_stock = pd.DataFrame(all_results)

elapsed = time.time() - t0
n_stock_cols = len(agg_stock.columns) - 1  # exclude date
print(f"\n  Aggregated in {elapsed:.1f}s")
print(f"  Result: {agg_stock.shape[0]:,} rows × {agg_stock.shape[1]} columns")
print(f"  Stock moment columns: {n_stock_cols} ({len(agg_factors)} factors × 5 moments)")

# Check for NaN
stock_moment_cols = [c for c in agg_stock.columns if c != 'date']
agg_nan = agg_stock[stock_moment_cols].isna().sum()
agg_nan_cols = agg_nan[agg_nan > 0]
if len(agg_nan_cols) > 0:
    print(f"\n  Columns with NaN after aggregation: {len(agg_nan_cols)}")
    for c in agg_nan_cols.sort_values(ascending=False).head(10).index:
        print(f"    {c}: {int(agg_nan_cols[c])} NaN")
else:
    print(f"  ✓ Zero NaN in aggregated stock moments")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 6: MERGE WITH MACRO DAILY + TARGET
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 6: MERGE AGGREGATED STOCK + MACRO DAILY + TARGET")
print("=" * 90)

# Check for column name conflicts
stock_cols_set = set(agg_stock.columns) - {'date'}
macro_cols_set = set(panel_c.columns) - {'date'}
overlap = stock_cols_set & macro_cols_set

if overlap:
    print(f"\n  ⚠ Column name conflicts ({len(overlap)}):")
    for c in sorted(list(overlap))[:10]:
        print(f"    {c}")
    if len(overlap) > 10:
        print(f"    ... and {len(overlap) - 10} more")
    print(f"    Adding 'stock_' prefix to conflicting stock columns...")
    rename_map = {c: f'stock_{c}' for c in overlap}
    agg_stock = agg_stock.rename(columns=rename_map)
    stock_moment_cols = [rename_map.get(c, c) for c in stock_moment_cols]
else:
    print(f"\n  ✓ No column name conflicts between stock moments and macro")

# Merge
result = agg_stock.merge(panel_c, on='date', how='inner')
result = result.merge(target_agg, on='date', how='left')

# Trim warmup rows (50 days for macro rolling features)
pre_trim = len(result)
result = result.iloc[50:].reset_index(drop=True)
print(f"\n  Trimmed first 50 rows for warmup: {pre_trim} → {len(result)}")

# Drop skew/kurt columns that are undefined because the cross-section is
# degenerate (cwstd ~ 0). Tested only from 2006-07 onward: before then some
# factors fall below MIN_STOCKS and are NaN for that reason instead, which is a
# late start rather than a degenerate cross-section. 2006-07 is where every
# factor first has adequate coverage, so this reproduces the original result
# exactly -- it is the only window the original ever saw.
DROP_TEST_START = '2008-01-01'
_m = result['date'] >= DROP_TEST_START

drop_undefined = [c for c in result.columns
                  if (c.endswith('_cwskew') or c.endswith('_cwkurt'))
                  and result.loc[_m, c].isna().any()]
result = result.drop(columns=drop_undefined)
print(f"Dropped {len(drop_undefined)} undefined skew/kurt columns "
      f"(tested from {DROP_TEST_START}):")
for c in drop_undefined:
    print(f"  {c}")

# Drop last row (no target — can't train on it)
result = result.dropna(subset=['target_daily_return']).reset_index(drop=True)
print(f"  Dropped last row (no next-day return): {len(result):,} rows")

print(f"  Merged result: {result.shape[0]:,} rows × {result.shape[1]} columns")
print(f"  Date range: {result['date'].min().date()} → {result['date'].max().date()}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 7: VALIDATE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 7: VALIDATE")
print("=" * 90)

# 7a. Duplicate dates
n_dupes = result['date'].duplicated().sum()
assert n_dupes == 0, "FATAL: Duplicate dates!"
print(f"\n  ✓ No duplicate dates")

# 7b. NaN in features
feature_cols = [c for c in result.columns if c not in ['date', 'target_daily_return']]
feature_nan = result[feature_cols].isna().sum()
feature_nan_total = feature_nan.sum()

if feature_nan_total > 0:
    nan_cols = feature_nan[feature_nan > 0].sort_values(ascending=False)
    print(f"\n  Feature NaN: {feature_nan_total}")
    print(f"  Columns with NaN ({len(nan_cols)}):")
    for c in nan_cols.head(15).index:
        print(f"    {c}: {int(nan_cols[c])}")
else:
    print(f"  ✓ Zero NaN in features")



# ── 7b2. Trailing NaN ───────────────────────────────────────────────────────
# A factor that stops publishing leaves the END of the sample empty, which lands
# in the test period. Split_D tests 2023-2024. The 5 discontinued OAP factors are
# dropped by name in notebooks 03/04, so anything listed here is new.
TRAILING_TOLERANCE = 126        # ~6 months of trading days

n_rows = len(result)
stopped = {}
for c in feature_cols:
    lv = result[c].last_valid_index()
    if lv is None:
        stopped[c] = ('never valid', n_rows)
    elif n_rows - 1 - lv > TRAILING_TOLERANCE:
        stopped[c] = (str(result['date'].iloc[lv].date()), n_rows - 1 - lv)

if stopped:
    print(f"\n  ** {len(stopped)} columns stop >{TRAILING_TOLERANCE} rows "
          f"before {result['date'].max().date()} **")
    print(f"  {'Column':<45s} {'Last valid':>12s} {'Rows missing':>13s}")
    for c, (d, g) in sorted(stopped.items(), key=lambda x: -x[1][1]):
        print(f"  {c:<45s} {d:>12s} {g:>13d}")
else:
    print(f"  ✓ No columns stop early")

# 7c. Target NaN
target_nan = result['target_daily_return'].isna().sum()
print(f"  Target NaN: {target_nan} (expect 1 — last trading day)")

# 7d. Target integrity
result_sorted = result.sort_values('date')
# Find the cwmean of dlyretx
dlyretx_cwmean = [c for c in result.columns if 'dlyretx_cwmean' in c]
if dlyretx_cwmean:
    col = dlyretx_cwmean[0]
    shifted = result_sorted[col].shift(-1)
    corr = result_sorted['target_daily_return'].corr(shifted)
    print(f"\n  Target integrity: corr(target_t, {col}_t+1) = {corr:.6f}")
    if corr > 0.95:
        print(f"  ✓ Target correctly represents next-day cap-weighted return")
    else:
        print(f"  ⚠ Correlation lower than expected — investigate!")

# 7e. Moment sanity checks
print(f"\n  Moment sanity checks:")

# cwstd should be positive
std_cols = [c for c in result.columns if c.endswith('_cwstd')]
if std_cols:
    min_std = result[std_cols].min().min()
    print(f"    Min cwstd across all factors/dates: {min_std:.8f} (should be ≥ 0)")

# cwkurt should be ≥ 1 (excess kurtosis ≥ -2 for any distribution)
kurt_cols = [c for c in result.columns if c.endswith('_cwkurt')]
if kurt_cols:
    min_kurt = result[kurt_cols].min().min()
    mean_kurt = result[kurt_cols].mean().mean()
    print(f"    Min cwkurt: {min_kurt:.4f}")
    print(f"    Mean cwkurt: {mean_kurt:.4f} (normal = 3.0, >3 = fat-tailed)")

# spread should be positive
spread_cols = [c for c in result.columns if c.endswith('_spread')
               and c.replace('_spread', '_cwmean') in result.columns]
if spread_cols:
    min_spread = result[spread_cols].min().min()
    print(f"    Min spread: {min_spread:.8f} (should be ≥ 0)")

# 7f. Column breakdown
stock_moment_count = len([c for c in result.columns if c in stock_moment_cols])
macro_count = len([c for c in result.columns if c in macro_factor_cols])
print(f"\n  Column breakdown:")
print(f"    Stock moment columns: {stock_moment_count} ({len(agg_factors)} factors × 5)")
print(f"    Macro factors:        {macro_count}")
print(f"    Target:               1")
print(f"    Date:                 1")
print(f"    Total:                {result.shape[1]}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 8: SAVE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 8: SAVE")
print("=" * 90)

result = result.sort_values('date').reset_index(drop=True)

out_path = OUT_DIR / 'agg_market_daily_full_moments.parquet'
result.to_parquet(out_path, index=False, engine='pyarrow')

file_size = out_path.stat().st_size
print(f"\n  ✓ Saved: {out_path}")
print(f"    {result.shape[0]:,} rows × {result.shape[1]} columns")
print(f"    Size: {file_size / 1e6:.1f} MB")

# ═══════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("DAILY FULL MOMENTS AGGREGATION COMPLETE")
print("=" * 90)

print(f"""
  Pipeline:
    Panel A ({panel_a.shape[0]:,} stock-days) → winsorise → 5 moments → {stock_moment_count} columns
    Panel C ({len(panel_c):,} days) → {macro_count} factors
    Target: next-day cap-weighted market return

  Moments per factor:
    cwmean  — cap-weighted average level
    cwstd   — cap-weighted cross-sectional dispersion
    cwskew  — cap-weighted cross-sectional skewness
    cwkurt  — cap-weighted cross-sectional kurtosis
    spread  — p90 minus p10 (unweighted range)

  Result:
    Rows:    {result.shape[0]:,} trading days
    Columns: {result.shape[1]}
    Dates:   {result['date'].min().date()} → {result['date'].max().date()}
    NaN:     {feature_nan_total} features + {target_nan} target (last day only)

  Saved: {out_path}

  Next: 03_build_agg_monthly_means.ipynb (Panel B + Panel D)
""")

STEP 1: LOAD & TRIM

  Panel A loaded: 525,957 rows × 196 columns
  After trim:  525,957 rows
  Date range: 2004-01-02 → 2024-12-31
  Unique dates: 5,285
  Avg stocks/date: 99.5

  Panel C loaded: 5,285 rows × 210 columns

  Panel A factor columns: 192
  Panel C factor columns: 209

STEP 2: HANDLE WARMUP NaN

  Factors with NaN: 187 / 192
  Total NaN cells: 4,126,860
  Strategy: NaN stocks excluded per-factor per-date during aggregation

STEP 3: WINSORISE STOCK FACTORS (1st/99th per date)

  Winsorised 192 factors in 23.9s
  ✓ Winsorisation complete
  Dropped 3 ISO columns

STEP 4: COMPUTE TARGET (next-day cap-weighted market return)

  Target computed: 5,284 dates
    Mean: 0.000470
    Std:  0.011776
  ✓ Last date (2024-12-31) correctly has no target

STEP 5: CAP-WEIGHTED FULL MOMENTS AGGREGATION

  Computing 5 statistics × 189 factors × 5,285 dates...
  (cwmean, cwstd, cwskew, cwkurt, spread)

    25/189 factors done... (13s elapsed, ~83s remaining)
    50/189 factors done... (23s e

In [2]:
# Check only the moment-computed spread columns, not macro factors
moment_spread_cols = [c for c in result.columns if c.endswith('_spread') 
                      and c.replace('_spread', '') + '_cwmean' in result.columns]
print(f"Min moment spread: {result[moment_spread_cols].min().min():.6f}")
print(f"Should be ≥ 0")

Min moment spread: 0.000000
Should be ≥ 0
